In [ ]:
# TCA (Transfer Component Analysis) - Basic Implementation

This is the basic TCA algorithm. See below for full PROMISE dataset implementation.

Source latent shape:0 200 5
Target latent shape: (500, 5)
Transformation matrix: 
 [[-0.13246463  0.00370403 -0.01803722 -0.00397019 -0.07820076]
 [-0.06287175  0.01115816 -0.04991301 -0.0265197   0.03550982]
 [ 0.09891635  0.05924271 -0.02931026 -0.10176175  0.07821987]
 [-0.05567829 -0.0275533  -0.0407341   0.0628653   0.02958996]
 [ 0.1454456   0.00417425 -0.01294135  0.00773764 -0.01271379]
 [ 0.04808944 -0.03430183  0.03863506  0.0696282  -0.08109685]
 [ 0.02728511  0.10205379  0.0845194  -0.07044822 -0.00186151]
 [-0.02802959 -0.13691997  0.00301409  0.05096488  0.15433896]
 [-0.01832136  0.10842708 -0.04106335  0.01320643  0.07597529]
 [ 0.05265942  0.00497824 -0.06274534 -0.04630355  0.04504415]
 [ 0.03963608 -0.13828595 -0.06969218  0.00244735  0.01382757]
 [-0.06241489 -0.0908147   0.01765761 -0.17001525 -0.03930881]
 [-0.01659522  0.04243421 -0.08732676 -0.08041201 -0.12016659]
 [-0.06481321  0.00534638  0.10217586  0.05421526 -0.06070898]
 [-0.01512764 -0.02211578  0.157637

In [ ]:
import numpy as np
from scipy.linalg import eigh

def construct_mmd_matrix(ns, nt):
    """
    Construct MMD matrix L
    """
    n = ns + nt
    L = np.zeros((n, n))

    # Source-source
    L[:ns, :ns] = 1.0 / (ns * ns)

    # Target-target
    L[ns:, ns:] = 1.0 / (nt * nt)

    # Source-target
    L[:ns, ns:] = -1.0 / (ns * nt)
    L[ns:, :ns] = -1.0 / (ns * nt)

    return L


def centering_matrix(n):
    """
    H = I - (1/n) * 11^T
    """
    return np.eye(n) - np.ones((n, n)) / n


def TCA(Xs, Xt, k=10, mu=1e-3):
    """
    Transfer Component Analysis (TCA)

    Parameters:
    ----------
    Xs : Source features (ns x d)
    Xt : Target features (nt x d)
    k  : Number of latent components
    mu : Regularization parameter

    Returns:
    -------
    Zs : Transformed source features
    Zt : Transformed target features
    W  : Transformation matrix
    """

    # 1️⃣ Stack data
    X = np.vstack((Xs, Xt))    # Stack them vertically (pile Xt on top of Xs)
    X = X.astype(float)    # #Converts the data type of array X to floating-point numbers.


    ns, nt = Xs.shape[0], Xt.shape[0]
    n, d = X.shape

    # 2️⃣ Normalize features (important!)
    X /= np.linalg.norm(X, axis=1, keepdims=True) + 1e-8
    #Computes the ℓ₂ norm (Euclidean length) of each row in matrix X.

    # 3️⃣ Construct matrices
    L = construct_mmd_matrix(ns, nt)
    H = centering_matrix(n)

    # 4️⃣ Compute matrices for eigen-problem
    A = X.T @ L @ X + mu * np.eye(d)
    B = X.T @ H @ X

    # 5️⃣ Solve generalized eigenvalue problem
    eigvals, eigvecs = eigh(A, B)

    # 6️⃣ Sort eigenvectors by smallest eigenvalues
    idx = np.argsort(eigvals)
    W = eigvecs[:, idx[:k]]

    # 7️⃣ Project data
    Z = X @ W
    Zs = Z[:ns]
    Zt = Z[ns:]

    return Zs, Zt, W


# Example shapes
# Xs: CM1 metrics
# Xt: JM1 metrics

Xs = np.random.rand(200, 20)   # source project
Xt = np.random.rand(500, 20)   # target project

Zs, Zt, W = TCA(Xs, Xt, k=5)

print("Source latent shape:", len(Zs), "x", len(Zs[0]))
print("Target latent shape:", Zt.shape)
print("Transformation matrix shape:", W.shape)

In [ ]:
# TCA Domain Adaptation for PROMISE Dataset

This notebook implements Transfer Component Analysis (TCA) for cross-project defect prediction using the PROMISE dataset.

## 1. Load PROMISE Dataset

We'll download and load PROMISE datasets (CM1, JM1, KC1, PC1, etc.) for cross-project defect prediction.

In [ ]:
def load_promise_dataset(dataset_name):
    """
    Load PROMISE dataset from OpenML repository
    
    Parameters:
    -----------
    dataset_name : str
        Name of the dataset (e.g., 'cm1', 'jm1', 'kc1', 'pc1')
        
    Returns:
    --------
    X : numpy array of features
    y : numpy array of labels (0: non-defective, 1: defective)
    """
    
    # PROMISE dataset URLs (NASA defect datasets)
    urls = {
        'cm1': 'https://raw.githubusercontent.com/klainfo/DefectData/master/NASA-MDP/CM1/CM1.csv',
        'jm1': 'https://raw.githubusercontent.com/klainfo/DefectData/master/NASA-MDP/JM1/JM1.csv',
        'kc1': 'https://raw.githubusercontent.com/klainfo/DefectData/master/NASA-MDP/KC1/KC1.csv',
        'kc2': 'https://raw.githubusercontent.com/klainfo/DefectData/master/NASA-MDP/KC2/KC2.csv',
        'pc1': 'https://raw.githubusercontent.com/klainfo/DefectData/master/NASA-MDP/PC1/PC1.csv',
        'mc2': 'https://raw.githubusercontent.com/klainfo/DefectData/master/NASA-MDP/MC2/MC2.csv',
    }
    
    if dataset_name.lower() not in urls:
        raise ValueError(f"Dataset {dataset_name} not found. Available: {list(urls.keys())}")
    
    # Load dataset
    df = pd.read_csv(urls[dataset_name.lower()])
    
    # Separate features and labels
    # The label column is usually the last column or named 'Defective'
    if 'Defective' in df.columns:
        y = df['Defective'].values
        X = df.drop('Defective', axis=1).values
    elif 'defects' in df.columns:
        y = df['defects'].values
        X = df.drop('defects', axis=1).values
    else:
        # Assume last column is the label
        y = df.iloc[:, -1].values
        X = df.iloc[:, :-1].values
    
    # Convert labels to binary (0: non-defective, 1: defective)
    y = (y > 0).astype(int)
    
    # Remove any non-numeric columns
    if isinstance(X, pd.DataFrame):
        X = X.select_dtypes(include=[np.number]).values
    
    print(f"✓ Loaded {dataset_name.upper()}: {X.shape[0]} instances, {X.shape[1]} features, {y.sum()} defects ({100*y.sum()/len(y):.1f}%)")
    
    return X, y


# Load multiple PROMISE datasets
print("Loading PROMISE Datasets...")
print("=" * 60)

## 2. Data Preprocessing

Handle missing values, align features, and normalize data.

## 3. TCA Implementation

The core TCA algorithm for domain adaptation.

## 4. Cross-Project Defect Prediction Evaluation

Compare performance with and without TCA domain adaptation.

In [ ]:
def evaluate_cpdp(X_source, y_source, X_target, y_target, use_tca=False, k=10):
    """
    Evaluate Cross-Project Defect Prediction (CPDP)
    
    Parameters:
    -----------
    X_source : Source project features
    y_source : Source project labels
    X_target : Target project features
    y_target : Target project labels
    use_tca : Whether to apply TCA domain adaptation
    k : Number of TCA components
    
    Returns:
    --------
    results : Dictionary of evaluation metrics
    """
    
    # Apply TCA if requested
    if use_tca:
        Zs, Zt, W = TCA(X_source, X_target, k=k, mu=1e-2)
        X_train = Zs
        X_test = Zt
    else:
        X_train = X_source
        X_test = X_target
    
    # Train classifier on source domain
    clf = LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced')
    clf.fit(X_train, y_source)
    
    # Predict on target domain
    y_pred = clf.predict(X_test)
    y_pred_proba = clf.predict_proba(X_test)[:, 1]
    
    # Calculate metrics
    results = {
        'accuracy': accuracy_score(y_target, y_pred),
        'precision': precision_score(y_target, y_pred, zero_division=0),
        'recall': recall_score(y_target, y_pred, zero_division=0),
        'f1': f1_score(y_target, y_pred, zero_division=0),
        'auc': roc_auc_score(y_target, y_pred_proba) if len(np.unique(y_target)) > 1 else 0.5
    }
    
    return results


def compare_methods(X_source, y_source, X_target, y_target, source_name, target_name):
    """
    Compare baseline vs TCA for cross-project defect prediction
    """
    print(f"\n{'='*70}")
    print(f"Cross-Project Defect Prediction: {source_name} → {target_name}")
    print(f"{'='*70}")
    
    # Baseline (no TCA)
    print("\n📊 Baseline (No Domain Adaptation):")
    baseline_results = evaluate_cpdp(X_source, y_source, X_target, y_target, use_tca=False)
    for metric, value in baseline_results.items():
        print(f"  {metric.capitalize():12s}: {value:.4f}")
    
    # TCA with different k values
    best_f1 = 0
    best_k = 5
    best_results = None
    
    for k in [5, 10, 15, 20]:
        results = evaluate_cpdp(X_source, y_source, X_target, y_target, use_tca=True, k=k)
        if results['f1'] > best_f1:
            best_f1 = results['f1']
            best_k = k
            best_results = results
    
    print(f"\n🚀 TCA Domain Adaptation (k={best_k}):")
    for metric, value in best_results.items():
        print(f"  {metric.capitalize():12s}: {value:.4f}")
    
    # Calculate improvement
    print(f"\n✨ Improvement:")
    for metric in baseline_results.keys():
        improvement = (best_results[metric] - baseline_results[metric]) / (baseline_results[metric] + 1e-8) * 100
        print(f"  {metric.capitalize():12s}: {improvement:+.2f}%")
    
    return baseline_results, best_results, best_k


print("✓ Evaluation functions defined")

In [ ]:
# Experiment 3: KC1 → PC1
X_kc1_p2, y_kc1_p2, X_pc1_p, y_pc1_p = preprocess_data(X_kc1, y_kc1, X_pc1, y_pc1)
baseline3, tca3, k3 = compare_methods(X_kc1_p2, y_kc1_p2, X_pc1_p, y_pc1_p, "KC1", "PC1")

## 5. Visualization

Visualize the performance comparison and domain distribution.

### Distribution Visualization

Visualize how TCA reduces the domain gap between source and target.

## 6. Summary Table

Summary of all experiments with improvement percentages.

## 7. Key Insights

**What is TCA?**
- Transfer Component Analysis (TCA) is a domain adaptation technique
- It finds a common latent space where source and target distributions are similar
- Minimizes Maximum Mean Discrepancy (MMD) between domains

**Why use TCA for Cross-Project Defect Prediction?**
- Different projects have different data distributions
- Models trained on one project (source) often perform poorly on another (target)
- TCA bridges this gap by aligning the distributions

**Key Benefits:**
1. ✅ Improved cross-project transfer learning
2. ✅ Better F1-score and recall for defect detection
3. ✅ Reduces domain shift between projects
4. ✅ No need for labeled data from target project

**Parameters:**
- `k`: Number of latent components (typically 5-20)
- `mu`: Regularization parameter (typically 1e-3 to 1e-1)

**When to use:**
- When you have labeled data from one project but want to predict on another
- When target project has different characteristics but similar features
- For cross-domain software defect prediction

In [ ]:
# Create summary table
summary_data = []

for i, exp in enumerate(experiments):
    baseline = baseline_results[i]
    tca = tca_results[i]
    
    for metric in metrics:
        improvement = (tca[metric] - baseline[metric]) / (baseline[metric] + 1e-8) * 100
        summary_data.append({
            'Experiment': exp,
            'Metric': metric.capitalize(),
            'Baseline': f"{baseline[metric]:.4f}",
            'TCA': f"{tca[metric]:.4f}",
            'Improvement': f"{improvement:+.2f}%"
        })

summary_df = pd.DataFrame(summary_data)

print("\n" + "="*80)
print("📊 SUMMARY: TCA Domain Adaptation Results")
print("="*80)
print(summary_df.to_string(index=False))
print("="*80)

# Calculate average improvements
print("\n📈 Average Improvements Across All Experiments:")
for metric in metrics:
    metric_improvements = []
    for i in range(len(experiments)):
        improvement = (tca_results[i][metric] - baseline_results[i][metric]) / (baseline_results[i][metric] + 1e-8) * 100
        metric_improvements.append(improvement)
    avg_improvement = np.mean(metric_improvements)
    print(f"  {metric.capitalize():12s}: {avg_improvement:+.2f}%")

In [ ]:
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

# Visualize CM1 → JM1 transfer
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Domain Distribution: CM1 (Source) → JM1 (Target)', fontsize=14, fontweight='bold')

# Before TCA
ax = axes[0]
X_combined = np.vstack([X_cm1_prep[:100], X_jm1_prep[:100]])  # Sample for speed
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_combined)

ax.scatter(X_pca[:100, 0], X_pca[:100, 1], c='red', label='CM1 (Source)', alpha=0.6, s=50)
ax.scatter(X_pca[100:, 0], X_pca[100:, 1], c='blue', label='JM1 (Target)', alpha=0.6, s=50)
ax.set_title('Before TCA (Original Space)', fontweight='bold')
ax.set_xlabel('PC1')
ax.set_ylabel('PC2')
ax.legend()
ax.grid(alpha=0.3)

# After TCA
ax = axes[1]
Zs_sample, Zt_sample, _ = TCA(X_cm1_prep[:100], X_jm1_prep[:100], k=2, mu=1e-2)

ax.scatter(Zs_sample[:, 0], Zs_sample[:, 1], c='red', label='CM1 (Source)', alpha=0.6, s=50)
ax.scatter(Zt_sample[:, 0], Zt_sample[:, 1], c='blue', label='JM1 (Target)', alpha=0.6, s=50)
ax.set_title('After TCA (Adapted Space)', fontweight='bold')
ax.set_xlabel('Component 1')
ax.set_ylabel('Component 2')
ax.legend()
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("\n✓ Visualization shows how TCA brings source and target distributions closer!")

In [ ]:
# Collect all results
experiments = ['CM1→JM1', 'JM1→KC1', 'KC1→PC1']
baseline_results = [baseline1, baseline2, baseline3]
tca_results = [tca1, tca2, tca3]
metrics = ['accuracy', 'precision', 'recall', 'f1', 'auc']

# Create comparison plots
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('TCA Domain Adaptation for Cross-Project Defect Prediction', fontsize=16, fontweight='bold')

# Plot each metric
for idx, metric in enumerate(metrics):
    if idx < 5:
        ax = axes[idx // 3, idx % 3]
        
        baseline_values = [res[metric] for res in baseline_results]
        tca_values = [res[metric] for res in tca_results]
        
        x = np.arange(len(experiments))
        width = 0.35
        
        bars1 = ax.bar(x - width/2, baseline_values, width, label='Baseline', alpha=0.8, color='#ff6b6b')
        bars2 = ax.bar(x + width/2, tca_values, width, label='TCA', alpha=0.8, color='#4ecdc4')
        
        ax.set_xlabel('Cross-Project Transfer', fontweight='bold')
        ax.set_ylabel(metric.capitalize(), fontweight='bold')
        ax.set_title(f'{metric.capitalize()} Comparison', fontweight='bold')
        ax.set_xticks(x)
        ax.set_xticklabels(experiments)
        ax.legend()
        ax.grid(axis='y', alpha=0.3)
        ax.set_ylim([0, 1.0])
        
        # Add value labels on bars
        for bars in [bars1, bars2]:
            for bar in bars:
                height = bar.get_height()
                ax.text(bar.get_x() + bar.get_width()/2., height,
                       f'{height:.3f}',
                       ha='center', va='bottom', fontsize=9)

# Remove empty subplot
axes[1, 2].axis('off')

plt.tight_layout()
plt.show()

print("\n" + "="*70)
print("📊 VISUALIZATION COMPLETE")
print("="*70)

### Experiment 3: KC1 → PC1

In [ ]:
# Experiment 2: JM1 → KC1
X_jm1_p, y_jm1_p, X_kc1_p, y_kc1_p = preprocess_data(X_jm1, y_jm1, X_kc1, y_kc1)
baseline2, tca2, k2 = compare_methods(X_jm1_p, y_jm1_p, X_kc1_p, y_kc1_p, "JM1", "KC1")

### Experiment 2: JM1 → KC1

In [ ]:
# Experiment 1: CM1 → JM1
baseline1, tca1, k1 = compare_methods(X_cm1_prep, y_cm1_prep, X_jm1_prep, y_jm1_prep, "CM1", "JM1")

### Experiment 1: CM1 → JM1

In [ ]:
def construct_mmd_matrix(ns, nt):
    """
    Construct Maximum Mean Discrepancy (MMD) matrix L
    
    This matrix measures the distance between source and target distributions.
    """
    n = ns + nt
    L = np.zeros((n, n))
    
    # Source-source: positive contribution
    L[:ns, :ns] = 1.0 / (ns * ns)
    
    # Target-target: positive contribution
    L[ns:, ns:] = 1.0 / (nt * nt)
    
    # Source-target: negative contribution (minimize distance)
    L[:ns, ns:] = -1.0 / (ns * nt)
    L[ns:, :ns] = -1.0 / (ns * nt)
    
    return L


def centering_matrix(n):
    """
    Construct centering matrix H = I - (1/n) * 11^T
    
    This centers the data (removes mean).
    """
    return np.eye(n) - np.ones((n, n)) / n


def TCA(Xs, Xt, k=10, mu=1e-3, kernel_type='linear'):
    """
    Transfer Component Analysis (TCA)
    
    Minimizes distribution distance while preserving important information.
    
    Parameters:
    ----------
    Xs : Source features (ns x d)
    Xt : Target features (nt x d)
    k  : Number of latent components (reduced dimensionality)
    mu : Regularization parameter (prevents overfitting)
    kernel_type : Type of kernel ('linear', 'rbf')
    
    Returns:
    -------
    Zs : Transformed source features (ns x k)
    Zt : Transformed target features (nt x k)
    W  : Transformation matrix (d x k)
    """
    
    # Stack source and target data
    X = np.vstack((Xs, Xt))
    X = X.astype(float)
    
    ns, nt = Xs.shape[0], Xt.shape[0]
    n, d = X.shape
    
    # Normalize features (important for stability)
    X_norm = X / (np.linalg.norm(X, axis=1, keepdims=True) + 1e-8)
    
    # Construct MMD matrix (measures domain distance)
    L = construct_mmd_matrix(ns, nt)
    
    # Construct centering matrix (removes mean)
    H = centering_matrix(n)
    
    # Compute matrices for generalized eigenvalue problem
    # A minimizes domain distance + regularization
    # B preserves data variance
    A = X_norm.T @ L @ X_norm + mu * np.eye(d)
    B = X_norm.T @ H @ X_norm + 1e-8 * np.eye(d)  # Add small term for numerical stability
    
    # Solve generalized eigenvalue problem: A*w = λ*B*w
    eigvals, eigvecs = eigh(A, B)
    
    # Select k eigenvectors with smallest eigenvalues
    # (minimize domain distance)
    idx = np.argsort(eigvals)
    W = eigvecs[:, idx[:k]]
    
    # Project data into new space
    Z = X_norm @ W
    Zs = Z[:ns]
    Zt = Z[ns:]
    
    return Zs, Zt, W


print("✓ TCA functions defined")

In [ ]:
def preprocess_data(X_source, y_source, X_target, y_target):
    """
    Preprocess source and target data for TCA
    
    - Handle missing values
    - Align features (use common features)
    - Normalize features
    """
    
    # Handle missing values
    X_source = np.nan_to_num(X_source, nan=0.0)
    X_target = np.nan_to_num(X_target, nan=0.0)
    
    # Ensure both have same number of features (use minimum)
    min_features = min(X_source.shape[1], X_target.shape[1])
    X_source = X_source[:, :min_features]
    X_target = X_target[:, :min_features]
    
    # Remove features with zero variance
    source_var = np.var(X_source, axis=0)
    target_var = np.var(X_target, axis=0)
    valid_features = (source_var > 1e-10) & (target_var > 1e-10)
    
    X_source = X_source[:, valid_features]
    X_target = X_target[:, valid_features]
    
    print(f"After preprocessing: {X_source.shape[1]} features")
    
    return X_source, y_source, X_target, y_target


# Example: Preprocess CM1 (source) and JM1 (target)
X_cm1_prep, y_cm1_prep, X_jm1_prep, y_jm1_prep = preprocess_data(X_cm1, y_cm1, X_jm1, y_jm1)

In [ ]:
# Load datasets
try:
    X_cm1, y_cm1 = load_promise_dataset('cm1')
    X_jm1, y_jm1 = load_promise_dataset('jm1')
    X_kc1, y_kc1 = load_promise_dataset('kc1')
    X_pc1, y_pc1 = load_promise_dataset('pc1')
    print("=" * 60)
    print("✓ All datasets loaded successfully!")
except Exception as e:
    print(f"Error loading datasets: {e}")

In [ ]:
import pandas as pd
import numpy as np
from scipy.linalg import eigh
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Set random seed for reproducibility
np.random.seed(42)